In [13]:
import pandas as pd

In [ ]:
paths = [
    ("datasets", "./data/Datasets.csv"),
    ("versions", "./data/DatasetVersions.csv"),
    ("dataset_tags", "./data/DatasetTags.csv"),
    ("tags", "./data/Tags.csv"),
]

for name, path in paths:
    df = pd.read_csv(path, nrows=3, low_memory=False)  # 仅读前3行，避免占内存
    print(f"\n== {name} | {path} ==")                  # 标注当前查看的表
    print("columns:", df.columns.tolist())             # 查看列名以识别主键/外键
    print(df.head(2))                                  # 打印两行样例，确认 id/datasetId/tagId 等


== datasets | ./data/Datasets.csv ==
columns: ['Id', 'CreatorUserId', 'OwnerUserId', 'OwnerOrganizationId', 'CurrentDatasetVersionId', 'CurrentDatasourceVersionId', 'ForumId', 'Type', 'CreationDate', 'LastActivityDate', 'TotalViews', 'TotalDownloads', 'TotalVotes', 'TotalKernels', 'Medal', 'MedalAwardDate']
       Id  CreatorUserId  OwnerUserId  OwnerOrganizationId  \
0  402034        2188609      2188609                  NaN   
1  402031        3792299      3792299                  NaN   

   CurrentDatasetVersionId  CurrentDatasourceVersionId  ForumId     Type  \
0                   771250                      792478   414197  Dataset   
1                   771247                      792475   414194  Dataset   

          CreationDate LastActivityDate  TotalViews  TotalDownloads  \
0  10/31/2019 01:07:32       10/31/2019        2706             128   
1  10/31/2019 00:58:15       10/31/2019        1795              37   

   TotalVotes  TotalKernels  Medal  MedalAwardDate  
0      

In [16]:
# 阶段 2a：检查 Datasets.csv 的版本指针质量（不做合并）
import pandas as pd

# 仅读取与检查有关的列，减小内存
ds = pd.read_csv("./data/Datasets.csv",
                 usecols=["Id", "CurrentDatasetVersionId", "CreationDate", "LastActivityDate"],
                 low_memory=False)

# 1) 规模与唯一性：Id 应与行数一致；若有重复说明主键异常
print("rows:", len(ds))                                  # 总行数
print("unique dataset Ids:", ds["Id"].nunique())         # 唯一的 Id 数
print("duplicate Ids:", ds["Id"].duplicated().sum())     # 统计重复 Id 的个数（期望为 0）

# 2) 当前版本指针缺失：若 >0，后续合并会出现 NaN 版本信息
print("null CurrentDatasetVersionId:",
      ds["CurrentDatasetVersionId"].isna().sum())

# 3) 当前版本指针复用：若 >0，表示有多个数据集指向同一 VersionId（通常不应发生）
print("duplicated CurrentDatasetVersionId:",
      ds["CurrentDatasetVersionId"].duplicated().sum())

# 4) 粗略时间一致性：创建时间不应晚于最近活动时间；>0 说明有异常时间记录
print("creation > last_activity count:",
      (pd.to_datetime(ds["CreationDate"], errors="coerce") >
       pd.to_datetime(ds["LastActivityDate"], errors="coerce")).sum())


rows: 521735
unique dataset Ids: 521735
duplicate Ids: 0
null CurrentDatasetVersionId: 245
duplicated CurrentDatasetVersionId: 244
creation > last_activity count: 517804


In [18]:
# 阶段 3｜标签映射与聚合（不引入行放大）
# 目标：
# 1) 用 Tags.csv 把 DatasetTags.csv 的 TagId 映射为可读的 TagName
# 2) 对每个 DatasetId 去重并聚合为一行（逗号分隔的标签字符串）

import pandas as pd

dt = pd.read_csv("./data/DatasetTags.csv", usecols=["DatasetId", "TagId"], low_memory=False)   # 只取必要列
tg = pd.read_csv("./data/Tags.csv", usecols=["Id", "Name"], low_memory=False)                  # 标签主表

tg = tg.rename(columns={"Id": "TagId", "Name": "TagName"})  # 对齐键名，便于合并
dt_named = dt.merge(tg, how="left", on="TagId")             # 映射 TagId -> TagName（左连接，保留所有打过标签的行）

dt_named = dt_named.drop_duplicates(subset=["DatasetId", "TagName"])  # 同一数据集重复的同名标签去重
tags_agg = (dt_named
            .groupby("DatasetId")["TagName"]                # 分组到每个数据集
            .apply(lambda s: ", ".join(sorted(s.dropna().astype(str))))  # 排序、去空、拼成逗号分隔
            .reset_index()
            .rename(columns={"TagName": "Tags"}))           # 得到：DatasetId, Tags

print(dt.shape, tg.shape, dt_named.shape, tags_agg.shape)   # 基本规模检查：聚合后行数应≈数据集中有标签的 DatasetId 数
print(tags_agg.head(3))                                     # 预览三行，确认 "DatasetId, Tags" 结构


(446054, 2) (831, 2) (446054, 3) (214603, 2)
   DatasetId                                            Tags
0          6  computer science, demographics, social science
1          7       internet, linguistics, online communities
2          8                atmospheric science, environment


In [19]:
# 将每个数据集的一行标签串（tags_agg）合并回 Datasets
import pandas as pd

ds = pd.read_csv("./data/Datasets.csv", low_memory=False)                     # 作为主表（保持一数据集一行）
ds_with_tags = ds.merge(tags_agg, how="left", left_on="Id", right_on="DatasetId")  # 按主键对齐，不引入行数放大
ds_with_tags = ds_with_tags.drop(columns=["DatasetId"])                       # 删除重复键列，避免“同样的列”

print("before/after shapes:", ds.shape, ds_with_tags.shape)                   # 行数应保持不变；列数多一列 Tags
print(ds_with_tags[["Id","Tags"]].head(3))                                    # 快速确认标签是否并入


before/after shapes: (521735, 16) (521735, 17)
       Id               Tags
0  402034                NaN
1  402031  health conditions
2   39875           bigquery


In [20]:
# 阶段 5｜填充缺失标签并保存结果（当前版本表已跳过）
# - 把没有标签的数据集的 Tags 填为空字符串，便于后续处理/检索
# - 将合并结果导出为 CSV 与 Parquet（若目录不存在则创建）

import os
os.makedirs("./output", exist_ok=True)                       # 确保输出目录存在
ds_with_tags["Tags"] = ds_with_tags["Tags"].fillna("")       # 将缺失标签填为空串，避免后续报错
ds_with_tags.to_csv("./output/datasets_with_tags.csv", index=False)
ds_with_tags.to_parquet("./output/datasets_with_tags.parquet", index=False)

print("saved:",
      "./output/datasets_with_tags.csv",
      "./output/datasets_with_tags.parquet")


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.